In [ ]:
import pandas as pd
import random
import time
import os
from datetime import datetime

# ✅ Create folder if it doesn't exist
os.makedirs("data/raw_stream", exist_ok=True)

file_path = "data/raw_stream/transactions_stream.csv"

# Sample users and transaction types
users = [f"user_{i}" for i in range(1, 101)]
transaction_types = ["PAYMENT", "TRANSFER", "CASH_OUT"]

# Generate one transaction
def generate_transaction():
    return {
        "timestamp": datetime.now(),
        "customer_id": random.choice(users),
        "amount": round(random.uniform(100, 500000), 2),
        "transaction_type": random.choice(transaction_types),
        "is_fraud": random.choice([0, 0, 0, 1])  # imbalanced fraud simulation
    }

# Stream transactions (LIMITED for testing)
def stream_transactions(n=20, delay=1):
    print(f"Starting stream for {n} transactions...\n")

    for i in range(n):
        txn = generate_transaction()
        df = pd.DataFrame([txn])

        # Write to CSV (append mode)
        df.to_csv(
            file_path,
            mode='a',
            header=not os.path.exists(file_path),
            index=False
        )

        print(f"{i+1}: New Transaction → {txn}")

        time.sleep(delay)

    print("\n✅ Streaming complete!")

# Run the stream
stream_transactions(n=20, delay=1)

Starting stream for 20 transactions...

1: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 30, 26, 954976), 'customer_id': 'user_54', 'amount': 138246.63, 'transaction_type': 'TRANSFER', 'is_fraud': 1}
2: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 30, 27, 961638), 'customer_id': 'user_92', 'amount': 418640.9, 'transaction_type': 'PAYMENT', 'is_fraud': 0}
3: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 30, 28, 965097), 'customer_id': 'user_97', 'amount': 110957.49, 'transaction_type': 'CASH_OUT', 'is_fraud': 0}
4: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 30, 29, 967949), 'customer_id': 'user_80', 'amount': 112914.54, 'transaction_type': 'CASH_OUT', 'is_fraud': 0}
5: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 30, 30, 970982), 'customer_id': 'user_28', 'amount': 248026.17, 'transaction_type': 'TRANSFER', 'is_fraud': 0}
6: New Transaction → {'timestamp': datetime.datetime(2026, 4, 

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw_stream/transactions_stream.csv")
df.head()

,2026-04-09 17:07:02.560077,user_15,21644.15,PAYMENT,0
0,2026-04-09 17:07:03.566910,user_29,259025.81,TRANSFER,0
1,2026-04-09 17:07:04.570010,user_76,488362.31,PAYMENT,0
2,2026-04-09 17:07:05.572856,user_99,346724.66,CASH_OUT,0
3,2026-04-09 17:07:06.576713,user_2,409529.45,PAYMENT,0
4,2026-04-09 17:07:07.581987,user_41,378118.28,TRANSFER,0


In [ ]:
stream_transactions(20)

Starting stream for 20 transactions...

1: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 46, 13, 794526), 'customer_id': 'user_67', 'amount': 202133.1, 'transaction_type': 'CASH_OUT', 'is_fraud': 0}
2: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 46, 14, 798744), 'customer_id': 'user_5', 'amount': 79354.17, 'transaction_type': 'TRANSFER', 'is_fraud': 0}
3: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 46, 15, 801207), 'customer_id': 'user_4', 'amount': 50549.92, 'transaction_type': 'CASH_OUT', 'is_fraud': 0}
4: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 46, 16, 804137), 'customer_id': 'user_78', 'amount': 348785.44, 'transaction_type': 'PAYMENT', 'is_fraud': 1}
5: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17, 46, 17, 807180), 'customer_id': 'user_26', 'amount': 41971.58, 'transaction_type': 'PAYMENT', 'is_fraud': 0}
6: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 17,

In [ ]:
!python streaming/stream_processor.py

In [ ]:
import pandas as pd
import time
import os

input_path = "data/processed/processed_transactions.csv"
output_path = "data/processed/scored_transactions.csv"

def calculate_risk(row):
    score = 0

    if row["amount"] > 200000:
        score += 2

    if row["avg_transaction_amount"] > 150000:
        score += 2

    if row["transaction_count"] > 3:
        score += 1

    if row["total_amount"] > 500000:
        score += 2

    return score

def run_scoring():
    print("Starting risk scoring...\n")

    last_processed_row = 0

    while True:
        if not os.path.exists(input_path):
            time.sleep(1)
            continue

        df = pd.read_csv(input_path)

        new_data = df.iloc[last_processed_row:]

        if new_data.empty:
            time.sleep(1)
            continue

        for _, row in new_data.iterrows():
            score = calculate_risk(row)
            risk_flag = 1 if score >= 4 else 0

            scored_row = row.to_dict()
            scored_row.update({
                "risk_score": score,
                "risk_flag": risk_flag
            })

            pd.DataFrame([scored_row]).to_csv(
                output_path,
                mode='a',
                header=not os.path.exists(output_path),
                index=False
            )

            print(f"Scored → User: {row['customer_id']} | Score: {score} | Flag: {risk_flag}")

        last_processed_row = len(df)

        time.sleep(1)

if __name__ == "__main__":
    run_scoring()

In [ ]:
stream_transactions(20)

Starting stream for 20 transactions...

1: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 21, 48, 23, 160790), 'customer_id': 'user_83', 'amount': 278622.69, 'transaction_type': 'PAYMENT', 'is_fraud': 1}
2: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 21, 48, 24, 164722), 'customer_id': 'user_73', 'amount': 425637.21, 'transaction_type': 'PAYMENT', 'is_fraud': 1}
3: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 21, 48, 25, 167753), 'customer_id': 'user_61', 'amount': 187888.56, 'transaction_type': 'PAYMENT', 'is_fraud': 0}
4: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 21, 48, 26, 180714), 'customer_id': 'user_48', 'amount': 184324.51, 'transaction_type': 'PAYMENT', 'is_fraud': 1}
5: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9, 21, 48, 27, 190442), 'customer_id': 'user_29', 'amount': 462497.82, 'transaction_type': 'CASH_OUT', 'is_fraud': 1}
6: New Transaction → {'timestamp': datetime.datetime(2026, 4, 9,

In [ ]:
!python streaming/stream_processor.py

python3: can't open file '/content/streaming/stream_processor.py': [Errno 2] No such file or directory


In [1]:
run_scoring()

NameError: name 'run_scoring' is not defined

In [7]:
import pandas as pd
import time
import os

input_path = "data/processed/processed_transactions.csv"
output_path = "data/processed/scored_transactions.csv"

def calculate_risk(row):
    score = 0

    if row["amount"] > 200000:
        score += 2

    if row["avg_transaction_amount"] > 150000:
        score += 2

    if row["transaction_count"] > 3:
        score += 1

    if row["total_amount"] > 500000:
        score += 2

    return score


def run_scoring_once():
    print("Running risk scoring (one-time)...\n")

    if not os.path.exists(input_path):
        print("No processed data found.")
        return

    df = pd.read_csv(input_path)

    results = []

    for _, row in df.iterrows():
        score = calculate_risk(row)
        risk_flag = 1 if score >= 4 else 0

        scored_row = row.to_dict()
        scored_row.update({
            "risk_score": score,
            "risk_flag": risk_flag
        })

        results.append(scored_row)

        print(f"Scored → User: {row['customer_id']} | Score: {score} | Flag: {risk_flag}")

    pd.DataFrame(results).to_csv(
        output_path,
        index=False
    )

    print("\n✅ Scoring complete!")

In [4]:
def run_scoring_once():
    print("Running risk scoring (one-time)...\n")

    if not os.path.exists(input_path):
        print("No processed data found.")
        return

    df = pd.read_csv(input_path)

    results = []

    for _, row in df.iterrows():
        score = calculate_risk(row)
        risk_flag = 1 if score >= 4 else 0

        scored_row = row.to_dict()
        scored_row.update({
            "risk_score": score,
            "risk_flag": risk_flag
        })

        results.append(scored_row)

        print(f"Scored → User: {row['customer_id']} | Score: {score} | Flag: {risk_flag}")

    pd.DataFrame(results).to_csv(
        output_path,
        index=False
    )

    print("\n✅ Scoring complete!")

In [8]:
def run_scoring_once():
    print("Running risk scoring (one-time)...\n")

    if not os.path.exists(input_path):
        print("No processed data found.")
        return

    df = pd.read_csv(input_path)

    results = []

    for _, row in df.iterrows():
        score = calculate_risk(row)
        risk_flag = 1 if score >= 4 else 0

        scored_row = row.to_dict()
        scored_row.update({
            "risk_score": score,
            "risk_flag": risk_flag
        })

        results.append(scored_row)

        print(f"Scored → User: {row['customer_id']} | Score: {score} | Flag: {risk_flag}")

    pd.DataFrame(results).to_csv(
        output_path,
        index=False
    )

    print("\n✅ Scoring complete!")

In [12]:
import os
os.makedirs("data/raw_stream", exist_ok=True)
print(os.listdir("data/raw_stream"))

[]


In [13]:
import pandas as pd
import random
import time
import os
from datetime import datetime

file_path = "/content/data/raw_stream/transactions_stream.csv"

users = [f"user_{i}" for i in range(1, 101)]
transaction_types = ["PAYMENT", "TRANSFER", "CASH_OUT"]

def generate_transaction():
    return {
        "timestamp": datetime.now(),
        "customer_id": random.choice(users),
        "amount": round(random.uniform(100, 500000), 2),
        "transaction_type": random.choice(transaction_types),
        "is_fraud": random.choice([0, 0, 0, 1])
    }

def stream_transactions(n=20):
    print("Streaming transactions...\n")

    for i in range(n):
        txn = generate_transaction()
        df = pd.DataFrame([txn])

        df.to_csv(
            file_path,
            mode='a',
            header=not os.path.exists(file_path),
            index=False
        )

        print(f"{i+1}: {txn}")

        time.sleep(0.5)

    print("\n✅ Done streaming!")

# Run it
stream_transactions(20)

Streaming transactions...

1: {'timestamp': datetime.datetime(2026, 4, 10, 9, 8, 53, 670571), 'customer_id': 'user_99', 'amount': 283580.85, 'transaction_type': 'CASH_OUT', 'is_fraud': 0}
2: {'timestamp': datetime.datetime(2026, 4, 10, 9, 8, 54, 192026), 'customer_id': 'user_26', 'amount': 70022.98, 'transaction_type': 'PAYMENT', 'is_fraud': 0}
3: {'timestamp': datetime.datetime(2026, 4, 10, 9, 8, 54, 694461), 'customer_id': 'user_4', 'amount': 476862.4, 'transaction_type': 'CASH_OUT', 'is_fraud': 0}
4: {'timestamp': datetime.datetime(2026, 4, 10, 9, 8, 55, 197234), 'customer_id': 'user_82', 'amount': 353414.44, 'transaction_type': 'CASH_OUT', 'is_fraud': 0}
5: {'timestamp': datetime.datetime(2026, 4, 10, 9, 8, 55, 699587), 'customer_id': 'user_80', 'amount': 200452.64, 'transaction_type': 'PAYMENT', 'is_fraud': 1}
6: {'timestamp': datetime.datetime(2026, 4, 10, 9, 8, 56, 201980), 'customer_id': 'user_10', 'amount': 494935.3, 'transaction_type': 'PAYMENT', 'is_fraud': 0}
7: {'timestamp

In [14]:
import pandas as pd

df = pd.read_csv("/content/data/raw_stream/transactions_stream.csv")
df.head()

,timestamp,customer_id,amount,transaction_type,is_fraud
0,2026-04-10 09:08:53.670571,user_99,283580.85,CASH_OUT,0
1,2026-04-10 09:08:54.192026,user_26,70022.98,PAYMENT,0
2,2026-04-10 09:08:54.694461,user_4,476862.40,CASH_OUT,0
3,2026-04-10 09:08:55.197234,user_82,353414.44,CASH_OUT,0
4,2026-04-10 09:08:55.699587,user_80,200452.64,PAYMENT,1


In [15]:
import pandas as pd
import os

input_path = "/content/data/raw_stream/transactions_stream.csv"
output_path = "/content/data/processed/processed_transactions.csv"

def process_once():
    print("Processing transactions...\n")

    if not os.path.exists(input_path):
        print("❌ No raw data found.")
        return

    df = pd.read_csv(input_path)

    user_state = {}
    results = []

    for _, row in df.iterrows():
        user = row["customer_id"]
        amount = row["amount"]

        if user not in user_state:
            user_state[user] = {
                "transaction_count": 0,
                "total_amount": 0
            }

        user_state[user]["transaction_count"] += 1
        user_state[user]["total_amount"] += amount

        txn_count = user_state[user]["transaction_count"]
        total_amt = user_state[user]["total_amount"]
        avg_amt = total_amt / txn_count

        processed_row = row.to_dict()
        processed_row.update({
            "transaction_count": txn_count,
            "total_amount": total_amt,
            "avg_transaction_amount": avg_amt
        })

        results.append(processed_row)

    os.makedirs("/content/data/processed", exist_ok=True)
    pd.DataFrame(results).to_csv(output_path, index=False)

    print("✅ Processing complete!")

# Run it
process_once()

Processing transactions...

✅ Processing complete!


In [16]:
df = pd.read_csv("/content/data/processed/processed_transactions.csv")
df.head()

,timestamp,customer_id,amount,transaction_type,is_fraud,transaction_count,total_amount,avg_transaction_amount
0,2026-04-10 09:08:53.670571,user_99,283580.85,CASH_OUT,0,1,283580.85,283580.85
1,2026-04-10 09:08:54.192026,user_26,70022.98,PAYMENT,0,1,70022.98,70022.98
2,2026-04-10 09:08:54.694461,user_4,476862.40,CASH_OUT,0,1,476862.40,476862.40
3,2026-04-10 09:08:55.197234,user_82,353414.44,CASH_OUT,0,1,353414.44,353414.44
4,2026-04-10 09:08:55.699587,user_80,200452.64,PAYMENT,1,1,200452.64,200452.64


In [17]:
import pandas as pd
import os

input_path = "/content/data/processed/processed_transactions.csv"
output_path = "/content/data/processed/scored_transactions.csv"

def calculate_risk(row):
    score = 0

    if row["amount"] > 200000:
        score += 2
    if row["avg_transaction_amount"] > 150000:
        score += 2
    if row["transaction_count"] > 3:
        score += 1
    if row["total_amount"] > 500000:
        score += 2

    return score

def run_scoring_once():
    print("Running risk scoring...\n")

    if not os.path.exists(input_path):
        print("❌ No processed data found.")
        return

    df = pd.read_csv(input_path)

    results = []

    for _, row in df.iterrows():
        score = calculate_risk(row)
        risk_flag = 1 if score >= 4 else 0

        scored_row = row.to_dict()
        scored_row.update({
            "risk_score": score,
            "risk_flag": risk_flag
        })

        results.append(scored_row)

        print(f"User: {row['customer_id']} | Score: {score} | Flag: {risk_flag}")

    pd.DataFrame(results).to_csv(output_path, index=False)

    print("\n✅ Scoring complete!")

# Run it
run_scoring_once()

Running risk scoring...

User: user_99 | Score: 4 | Flag: 1
User: user_26 | Score: 0 | Flag: 0
User: user_4 | Score: 4 | Flag: 1
User: user_82 | Score: 4 | Flag: 1
User: user_80 | Score: 4 | Flag: 1
User: user_10 | Score: 4 | Flag: 1
User: user_13 | Score: 4 | Flag: 1
User: user_25 | Score: 0 | Flag: 0
User: user_9 | Score: 0 | Flag: 0
User: user_87 | Score: 0 | Flag: 0
User: user_77 | Score: 2 | Flag: 0
User: user_60 | Score: 4 | Flag: 1
User: user_37 | Score: 0 | Flag: 0
User: user_95 | Score: 4 | Flag: 1
User: user_62 | Score: 0 | Flag: 0
User: user_100 | Score: 4 | Flag: 1
User: user_77 | Score: 6 | Flag: 1
User: user_52 | Score: 0 | Flag: 0
User: user_59 | Score: 4 | Flag: 1
User: user_30 | Score: 4 | Flag: 1

✅ Scoring complete!


In [18]:
df = pd.read_csv("/content/data/processed/scored_transactions.csv")
df.head()

,timestamp,customer_id,amount,transaction_type,is_fraud,transaction_count,total_amount,avg_transaction_amount,risk_score,risk_flag
0,2026-04-10 09:08:53.670571,user_99,283580.85,CASH_OUT,0,1,283580.85,283580.85,4,1
1,2026-04-10 09:08:54.192026,user_26,70022.98,PAYMENT,0,1,70022.98,70022.98,0,0
2,2026-04-10 09:08:54.694461,user_4,476862.40,CASH_OUT,0,1,476862.40,476862.40,4,1
3,2026-04-10 09:08:55.197234,user_82,353414.44,CASH_OUT,0,1,353414.44,353414.44,4,1
4,2026-04-10 09:08:55.699587,user_80,200452.64,PAYMENT,1,1,200452.64,200452.64,4,1


In [19]:
alert_path = "/content/data/processed/fraud_alerts.csv"

In [20]:
import pandas as pd
import os

input_path = "/content/data/processed/scored_transactions.csv"
alert_path = "/content/data/processed/fraud_alerts.csv"

def generate_alerts():
    print("Generating fraud alerts...\n")

    if not os.path.exists(input_path):
        print("❌ No scored data found.")
        return

    df = pd.read_csv(input_path)

    alerts = df[df["risk_flag"] == 1]

    if alerts.empty:
        print("✅ No fraud alerts detected.")
        return

    alerts.to_csv(alert_path, index=False)

    print(f"🚨 {len(alerts)} fraud alerts generated!")

# Run it
generate_alerts()

Generating fraud alerts...

🚨 12 fraud alerts generated!


In [21]:
df = pd.read_csv("/content/data/processed/fraud_alerts.csv")
df.head()

,timestamp,customer_id,amount,transaction_type,is_fraud,transaction_count,total_amount,avg_transaction_amount,risk_score,risk_flag
0,2026-04-10 09:08:53.670571,user_99,283580.85,CASH_OUT,0,1,283580.85,283580.85,4,1
1,2026-04-10 09:08:54.694461,user_4,476862.40,CASH_OUT,0,1,476862.40,476862.40,4,1
2,2026-04-10 09:08:55.197234,user_82,353414.44,CASH_OUT,0,1,353414.44,353414.44,4,1
3,2026-04-10 09:08:55.699587,user_80,200452.64,PAYMENT,1,1,200452.64,200452.64,4,1
4,2026-04-10 09:08:56.201980,user_10,494935.30,PAYMENT,0,1,494935.30,494935.30,4,1


In [28]:
from google.colab import files
import os

# IMPORTANT: Please save your notebook manually before running this cell.
# If you intend to save it to 'streaming/transaction_simulator.ipynb' in Colab,
# ensure the 'streaming' directory exists or save it to the root directory.

notebook_filename = 'streaming/transaction_simulator.ipynb' # This is the name you want to download it as

# Ensure the target directory exists in Colab if you manually saved it there.
# This is for the *Colab file system path* where the notebook is expected to be saved.
# For example, if you saved it as /content/streaming/transaction_simulator.ipynb
os.makedirs(os.path.dirname(notebook_filename), exist_ok=True)

try:
    # Attempt to download the file that you manually saved in the Colab environment
    if not os.path.exists(notebook_filename):
        print(f"❌ Error: The file '{notebook_filename}' was not found in the Colab environment.")
        print("Please ensure you have manually saved your notebook with this exact path and filename.")
    else:
        print(f"Attempting to download '{notebook_filename}'...")
        files.download(notebook_filename)
        print("✅ Notebook download initiated!")

except Exception as e:
    print(f"❌ An unexpected error occurred during download: {e}")
    print("Please verify your manual save location and filename in Colab.")

❌ Error: The file 'streaming/transaction_simulator.ipynb' was not found in the Colab environment.
Please ensure you have manually saved your notebook with this exact path and filename.
